In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%load_ext lab_black
# or nb_black

# Idea

Using the already implemented `ARLLMLLMRecommender`, `LLMGenAIRecommender` will call Gemini API to reach the final selection of recommendations.

In [63]:
from ecommercerecommendation.utils.constants import PROMPT_TEMPLATE
from ecommercerecommendation.utils.prompts import Prompt

In [64]:
prompt = Prompt(PROMPT_TEMPLATE)

In [65]:
# print(PROMPT_TEMPLATE)

In [66]:
selection = ["REGENCY TEA PLATE ROSES", "REGENCY TEA PLATE PINK"]

In [67]:
candidates = [
    " ".join(c.split(" ")[1:])
    for c in [
        "20674 GREEN POLKADOT BOWL",
        "21531 RED RETROSPOT SUGAR JAM BOWL",
        "21533 RETROSPOT LARGE MILK JUG",
        "21534 DAIRY MAID LARGE MILK JUG",
        "21535 RED RETROSPOT SMALL MILK JUG",
        "22072 RED RETROSPOT TEA CUP AND SAUCER",
        "22697 GREEN REGENCY TEACUP AND SAUCER",
        "22698 PINK REGENCY TEACUP AND SAUCER",
        "22776 SWEETHEART CAKESTAND 3 TIER",
        "22890 NOVELTY BISCUITS CAKE STAND 3 TIER",
        "23160 REGENCY TEA SPOON",
        "23161 REGENCY CAKE FORK",
        "23163 REGENCY SUGAR TONGS",
        "23164 REGENCY CAKE SLICE",
        "23170 REGENCY TEA PLATE ROSES",
        "23171 REGENCY TEA PLATE GREEN",
        "23172 REGENCY TEA PLATE PINK",
        "23173 REGENCY TEAPOT ROSES",
        "23245 SET OF 3 REGENCY CAKE TINS",
    ]
]

In [68]:
from ecommercerecommendation.utils.constants import BULLET_POINT

In [69]:
prompt.format(
    {
        "top_n": 5,
        "selected": BULLET_POINT.join(selection),
        "recommendation_candidates": BULLET_POINT.join(candidates),
    }
)

In [70]:
print(prompt.__prompt__)

<role>You are part of a bundle recommender system. You are given a list of already selected products by the user, and also a list of candidates to be recommended to buy together.</role>

<task>Your task is to select those elements of the candidate list, which are the most probable to be bought together, knowing what the user have already selected. 
The user has already selected the following products:
- "REGENCY TEA PLATE ROSES
- REGENCY TEA PLATE PINK"
Output is in JSON format with one key: “recommendations” (list of strings), such as:
{
    "recommendations": [],
}

You are a JSON generator. Return only valid JSON.</task>

<candidate_list>
- "GREEN POLKADOT BOWL
- RED RETROSPOT SUGAR JAM BOWL
- RETROSPOT LARGE MILK JUG
- DAIRY MAID LARGE MILK JUG
- RED RETROSPOT SMALL MILK JUG
- RED RETROSPOT TEA CUP AND SAUCER
- GREEN REGENCY TEACUP AND SAUCER
- PINK REGENCY TEACUP AND SAUCER
- SWEETHEART CAKESTAND 3 TIER
- NOVELTY BISCUITS CAKE STAND 3 TIER
- REGENCY TEA SPOON
- REGENCY CAKE FORK
-

# Imports and data

In [3]:
import pickle
import json
from ecommercerecommendation.utils.data import get_data, venn_sets

# from ecommercerecommendation.models.arllmllmrecommender import LLMGenAIRecommender

In [4]:
df = get_data("clean_data")

In [5]:
print("Columns: " + ", ".join(df.columns))

Columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country


## Train-test-split

We will use a 80-20 split by InvoiceDate in order to prevent data leakage.

In [6]:
cut_off_date = df["InvoiceDate"].quantile(0.8)
df["train"] = df["InvoiceDate"] <= cut_off_date

In [7]:
X_train = df[df["train"]].drop(columns="train")
X_test = df[~df["train"]].drop(columns="train")

## LLM-GenAI algorithm

In [78]:
# llmgenai_recommender = LLMGenAIRecommender()

In [79]:
# llmgenai_recommender.fit(X_train)

In [80]:
# selection3 = ["REGENCY TEA PLATE ROSES", "REGENCY TEA PLATE PINK"]

In [81]:
ar_llm_llm_results = [
    "20674 GREEN POLKADOT BOWL",
    "21531 RED RETROSPOT SUGAR JAM BOWL",
    "21533 RETROSPOT LARGE MILK JUG",
    "21534 DAIRY MAID LARGE MILK JUG",
    "21535 RED RETROSPOT SMALL MILK JUG",
    "22072 RED RETROSPOT TEA CUP AND SAUCER",
    "22697 GREEN REGENCY TEACUP AND SAUCER",
    "22698 PINK REGENCY TEACUP AND SAUCER",
    "22776 SWEETHEART CAKESTAND 3 TIER",
    "22890 NOVELTY BISCUITS CAKE STAND 3 TIER",
    "23160 REGENCY TEA SPOON",
    "23161 REGENCY CAKE FORK",
    "23163 REGENCY SUGAR TONGS",
    "23164 REGENCY CAKE SLICE",
    "23170 REGENCY TEA PLATE ROSES",
    "23171 REGENCY TEA PLATE GREEN",
    "23172 REGENCY TEA PLATE PINK",
    "23173 REGENCY TEAPOT ROSES",
    "23245 SET OF 3 REGENCY CAKE TINS",
]

In [82]:
# llmgenai_recommender.get_recommendations(
#     current_selection=selection3,
# )